# Fase 1: limpieza de datos

Este notebook sirve para **explorar y probar** la limpieza con los datos simulados
(`data/simulated/`). Cuando una etapa funcione, su código se pasa a
`src/occupancy/cleaning.py`.

Orden de las etapas (ver el documento del pipeline):

0. Cargar los datos y convertir las fechas
1. Duplicados
2. Control de rango
3. Picos aislados
4. Huecos
5. Suavizado (media móvil)

Las celdas marcadas como **Comprobación** ya están escritas: ejecútalas después de
tu código para saber si el resultado es correcto.

In [ ]:
import json
from pathlib import Path

import pandas as pd
import yaml

# Buscar la raíz del repositorio (la carpeta que contiene config.yaml)
ROOT = Path.cwd()
while not (ROOT / "config.yaml").exists():
    ROOT = ROOT.parent

# Leer la configuración (después de encontrar la raíz)
with open(ROOT / "config.yaml", encoding="utf-8") as f:
    config = yaml.safe_load(f)
cleaning = config["cleaning"]

DATA_PATH = ROOT / "data" / "simulated" / "readings.json"
FAULTS_PATH = ROOT / "data" / "simulated" / "faults.json"
print("Datos:", DATA_PATH)

## 0. Cargar los datos

El archivo tiene el mismo formato que la respuesta de la API. Esta celda ya lo lee y
deja en `data` un diccionario con una lista de lecturas por sensor.

In [ ]:
with open(DATA_PATH, encoding="utf-8") as f:
    response = json.load(f)

data = response["data"]
print("Sensores:", list(data))
print("Primera lectura de temp:", data["temperature"][0])
print("Primera lectura de PIR:", data["pirs"][0])

### Tu tarea: convertir cada sensor en una tabla

Guarda el resultado en un diccionario llamado **`tables`**: una tabla por sensor
(menos `pirs` y la etiqueta `busynessLabel`), con las columnas `timestamp` y `value`.

- Cada lista de lecturas se convierte en tabla con `pd.DataFrame`.
- Convierte `timestamp` a fecha con `pd.to_datetime`, indicando el formato
  `"%Y-%m-%d %H:%M:%S.%f"`.

In [ ]:
# Tu código aquí
tables = {}
for sensor, readings in data.items():
    if sensor in ("pirs", "busynessLabel"):
        continue
    table = pd.DataFrame(readings)
    table["timestamp"] = pd.to_datetime(table["timestamp"], format="%Y-%m-%d %H:%M:%S.%f")
    tables[sensor] = table

for i, j in tables.items():
    print(f"Tabla de {i}:")
    print(j.head())


### Tu tarea: convertir los PIR en una tabla

Guarda el resultado en una tabla llamada **`pir`**, con una fila por marca de tiempo y
las columnas `timestamp`, `pir-1`, `pir-2` y `pir-3`. Convierte también `timestamp`
a fecha.

In [ ]:
# Tu código aquí
rows = []
for reading in data["pirs"]:
    row = {"timestamp": pd.to_datetime(reading["timestamp"], format="%Y-%m-%d %H:%M:%S.%f")}
    for p in reading["values"]:
        # columna "pir-<ID>" con la actividad del PIR (0 a 100)
        row[f"pir-{p['ID']}"] = p["activity"]
    # añade row a la lista rows
    rows.append(row)

pir = pd.DataFrame(rows)
print(pir.head())
# convierte la columna timestamp a fecha

In [ ]:
# Comprobación de la carga (no hace falta modificarla)
expected = {s: 1067 for s in ["pressure", "humidity", "temperature", "co2Equivalent", "voc", "iaq", "mic_vol", "wifiPackets", "bleAdverts"]}

for sensor, n in expected.items():
    table = tables[sensor]
    ok = len(table) == n and pd.api.types.is_datetime64_any_dtype(table["timestamp"])
    print(f"{sensor:14} filas={len(table):5}  esperado={n:5}  {'OK' if ok else 'REVISAR'}")

ok_pir = (len(pir) == 1067
          and set(pir.columns) == {"timestamp", "pir-1", "pir-2", "pir-3"}
          and pd.api.types.is_datetime64_any_dtype(pir["timestamp"]))
print(f"{'PIR':14} filas={len(pir):5}  esperado= 1067  {'OK' if ok_pir else 'REVISAR'}")

## 1.1 Duplicados

Un duplicado de QoS 1 es una fila **exactamente igual** a otra: misma hora y mismos
valores. Quítalos con `drop_duplicates` sobre la fila completa, no solo sobre la hora.

Guarda el resultado en las mismas variables (`tables` y `pir`).

In [ ]:
# Tu código aquí
for sensor, table in tables.items():
    tables[sensor] = table.drop_duplicates()

pir= pir.drop_duplicates()

In [ ]:
# Comprobación de duplicados (no hace falta modificarla)
expected = {s: 1059 for s in ["pressure", "humidity", "temperature", "co2Equivalent", "voc", "iaq", "mic_vol", "wifiPackets", "bleAdverts"]}

for sensor, n in expected.items():
    table = tables[sensor]
    ok = len(table) == n and table.duplicated().sum() == 0
    print(f"{sensor:14} filas={len(table):5}  esperado={n:5}  {'OK' if ok else 'REVISAR'}")

ok_pir = len(pir) == 1059 and pir.duplicated().sum() == 0
print(f"{'PIR':14} filas={len(pir):5}  esperado= 1059  {'OK' if ok_pir else 'REVISAR'}")

In [ ]:
for sensor, table in tables.items():
    print(sensor, type(table).__name__)

## 1.2 Control de rango

Convierte en vacío (`NaN`) los valores imposibles para cada sensor, **sin borrar la
fila**. Los límites reales saldrán de la hoja de datos de cada sensor; para los datos
simulados basta con detectar los valores claramente imposibles.

Herramienta: `mask`.

In [ ]:
# Tu código aquí
Limits = cleaning["limits"]

for sensor, table in tables.items():
    limits = Limits[sensor]
    condition = (table["value"] < limits["min"]) | (table["value"] > limits["max"])
    tables[sensor]["value"]  = table["value"].mask(condition)

    



## 1.3 Picos aislados

Convierte en vacío los valores que se disparan un instante y vuelven, comparando cada
valor con sus vecinos. Sobre todo en `mic_vol`.

Herramientas: `rolling` y `mask`.

In [ ]:
spike_thresholds = cleaning["spike_thresholds"]

for sensor, threshold in spike_thresholds.items():
    table = tables[sensor]
    median = table["value"].rolling(cleaning["spike_window_readings"], center=True, min_periods=1).median()
    distance = table["value"].sub(median).abs()
    spikes = distance > threshold
    print(sensor, "picos detectados:", spikes.sum())
    table["value"] = table["value"].mask(spikes)

## 1.4 Huecos

Primero mide cada cuánto llegan las lecturas (`diff` sobre `timestamp`). Después
rellena solo los huecos cortos y deja vacíos los largos.

Herramienta: `interpolate` con `method="time"` y un límite.

In [ ]:
max_gap = cleaning["max_gap_readings"]   # máximo de lecturas seguidas que se rellenan

# Intervalo común: todos los sensores llegan en el mismo mensaje
interval = pd.Timedelta(seconds=cleaning["interval_seconds"])
print("Intervalo común:", interval)

for sensor in tables:
    table = tables[sensor]

    # Comprobación: el intervalo de este sensor coincide con el común
    own_interval = table["timestamp"].diff().median()
    if own_interval != interval:
        print("AVISO:", sensor, "tiene otro intervalo:", own_interval)

    # Rejilla regular con el intervalo común
    series = table.set_index("timestamp")["value"].resample(interval).mean()

    # Medir la longitud de cada hueco
    empty = series.isna()
    group = (empty != empty.shift()).cumsum()
    gap_length = empty.groupby(group).transform("sum")
    short_gap = empty & (gap_length <= max_gap)

    # Interpolar y quedarse solo con lo que cae en huecos cortos
    filled = series.interpolate(method="time", limit_area="inside")
    series = series.mask(short_gap, filled)

    print(sensor, "vacíos:", empty.sum(), "rellenados:", short_gap.sum(), "quedan:", series.isna().sum())

    tables[sensor] = series.reset_index()


# Rejilla regular con el mismo intervalo que el resto de sensores
grid = pir.set_index("timestamp").resample(interval).mean()

for column in ["pir-1", "pir-2", "pir-3"]:
    series = grid[column]

    # Medir la longitud de cada hueco
    # (las mismas cuatro líneas que en la etapa de huecos)
    empty = grid[column].isna()
    group = (empty != empty.shift()).cumsum()
    gap_length = empty.groupby(group).transform("sum")
    short_gap = empty & (gap_length <= max_gap)
    # Interpolar y quedarse solo con lo que cae en huecos cortos
    # (las mismas dos líneas que en la etapa de huecos)
    filled = series.interpolate(method="time", limit_area="inside")
    series = series.mask(short_gap, filled)
    print(column, "vacíos:", empty.sum(), "rellenados:", short_gap.sum(), "quedan:", series.isna().sum())

    # Guarda la columna corregida en grid
    grid[column] = series

pir = grid.reset_index()

## 1.5 Suavizado

Media móvil que **solo mira hacia atrás**, aplicada a `mic_vol`, `co2Equivalent` y `voc`.

Herramienta: `rolling` con una ventana de tiempo.

In [ ]:
# Tu código aquí
apply_smoothing = cleaning["smoothing"]["enabled"]
smooth_window = pd.Timedelta(seconds=cleaning["smoothing"]["window_seconds"])

if apply_smoothing:
    for sensor in cleaning["smoothing"]["sensors"]:
        table = tables[sensor]

        # Pon timestamp como índice y quédate con la columna value
        series = table.set_index("timestamp")["value"]

        # Media móvil de los últimos 30 s
        smoothed = series.rolling(smooth_window, min_periods=1).mean()

        # Vuelve a dejar vacío lo que ya estaba vacío
        smoothed = smoothed.where(series.notna())

        print(sensor, "ruido antes:", series.diff().std(), "después:", smoothed.diff().std())
        print(sensor, "vacíos:", smoothed.isna().sum())

        # Vuelve a la forma de tabla y guárdala en tables
        tables[sensor] = smoothed.reset_index(name="value")

## Inspección visual

Dibuja cada sensor antes y después de la limpieza para comprobar que no queda nada raro.
Es la parte de "inspección humana" de la diapositiva 25.

In [ ]:
import matplotlib.pyplot as plt

# ---------- Gráfica 1: mic_vol antes y después de la limpieza ----------
sensor = "mic_vol"

# Antes: los datos originales, que siguen intactos en data
raw = pd.DataFrame(data[sensor])
raw["timestamp"] = pd.to_datetime(raw["timestamp"], format="%Y-%m-%d %H:%M:%S.%f")

# Después: la tabla ya limpia
clean = tables[sensor]

fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)

axes[0].plot(raw["timestamp"], raw["value"], color="tab:red", linewidth=0.8)
axes[0].set_title(f"{sensor}: antes de la limpieza")
axes[0].set_ylabel("valor")

axes[1].plot(clean["timestamp"], clean["value"], color="tab:blue", linewidth=0.8)
axes[1].set_title(f"{sensor}: después de la limpieza")
axes[1].set_ylabel("valor")
axes[1].set_xlabel("hora")

plt.tight_layout()
plt.show()

# ---------- Gráfica 2: todos los sensores ya limpios ----------
sensors = list(tables)
fig, axes = plt.subplots(len(sensors), 1, figsize=(12, 2 * len(sensors)), sharex=True)

for ax, sensor in zip(axes, sensors):
    ax.plot(tables[sensor]["timestamp"], tables[sensor]["value"], linewidth=0.8)
    ax.set_title(sensor, loc="left")

axes[-1].set_xlabel("hora")

plt.tight_layout()
plt.show()
